In [ ]:
#Importing datasets
import pickle
import numpy
import pandas
from pandas import read_csv
from collections import Counter
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import RidgeClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import GradientBoostingClassifier
from scipy.stats import uniform
from sklearn.model_selection import train_test_split
#Dataset used - Titanic Survivors Dataset taken from Kaggle
T_Train= pandas.read_csv("/kaggle/input/titanic/train.csv")
T_Test= pandas.read_csv("/kaggle/input/titanic/test.csv")
T_Train.head()



In [ ]:
T_Train.set_index('PassengerId')
T_Train.shape

In [ ]:
T_Test.set_index('PassengerId')
T_Test.shape

In [ ]:
T_Concat=pandas.concat([T_Train,T_Test],axis=0,sort=False)
T_Concat.tail()

In [ ]:
T_Concat.describe()

In [ ]:
T_Concat.info()

In [ ]:
#Imputing missing Values
T_Concat.isnull().sum()

In [ ]:
#There are missing values  in Age, Fare, Cabin and Embarked Variables
#Lets calculate missing values in Age first, as Survived is the outcome variable and missing values  are for Test Variable
T_Concat.Age.describe()

In [ ]:
#We can impute  missing values  with the mean value using simple imputer
Age_Mean=T_Concat.Age.mean()
Age_Mean
T_Concat.loc[T_Concat['Age'].isnull(),'Age']=Age_Mean
T_Concat.isnull().sum()

In [ ]:
#Similarly we can impute missing value  of Fare using mean value
Fare_Mean=T_Concat.Fare.mean()
Fare_Mean
T_Concat.loc[T_Concat['Fare'].isnull(),'Fare']=Fare_Mean
T_Concat.isnull().sum()

In [ ]:
#There is no relevance of Cabin number  for relation with results  as  most of the values  are missing.  
#Moreover, cabin numbers  are unique  as there are  186 unique items. So we can remove Cabin feature.
T_Concat.Cabin.nunique()

In [ ]:
T_Concat1=T_Concat.drop('Cabin',axis=1)
T_Concat1.head()

In [ ]:
T_Concat1.isnull().sum()

In [ ]:
#Imputing missing values  for  Embarked variable
T_Concat1.Embarked.value_counts()

In [ ]:
T_Concat1.Embarked=T_Concat1.Embarked.astype('category')
T_Concat1.isnull().sum()

In [ ]:
T_Concat1.loc[T_Concat1['Embarked'].isnull(),'Embarked']='S'
T_Concat1.isnull().sum()

In [ ]:
#Now  all the missing values  are imputed.
T_Concat1.describe()

In [ ]:
T_Concat1.Embarked.describe()

In [ ]:
T_Concat1.head()

In [ ]:
T_Concat1=T_Concat1.set_index('PassengerId')
T_Concat1.head()

In [ ]:
#Lets us check co-relation between the variables
import seaborn
seaborn.heatmap(T_Concat1.corr())

In [ ]:
#Heat map  is not reflecting relationship for objects(strings)  and categorical variables
##Let us analyze variable Sex and try to convert to integer
T_Concat1.info()

In [ ]:
T_Concat1.Sex=T_Concat1.Sex.astype('category')
T_Concat1.info()

In [ ]:
#Now  we would use One Hot Encoding to convert variable Sex and Embarked from categorical to integer type
#We have already converted variable Embarked to category
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder(sparse=False)
Age_D=T_Concat1.loc[:,lambda T_Concat1: T_Concat1.dtypes == 'category']
Age_D.head()

In [ ]:
Age_T=encoder.fit_transform(Age_D)
Age_T

In [ ]:
Age_Df=pandas.DataFrame(Age_T, columns=['Female','Male','C','Q','S'])
Age_Df.info()

In [ ]:
Age_Df.head()

In [ ]:
#Now we need to correct the indexing of transformed dataframe to match the existing Dataframe

Age_Df=Age_Df.set_index(T_Concat1.index)
Age_Df.head()
T_Concat2=pandas.concat([T_Concat1,Age_Df],axis=1,sort=False)
T_Concat2.info()
T_Concat2.head()

In [ ]:
#Now we an remove the categorical variables from dataset
T_Concat2=T_Concat2.drop('Sex',axis=1)
T_Concat2=T_Concat2.drop('Embarked',axis=1)
T_Concat2.head()

In [ ]:
T_Concat2.info()

In [ ]:
#Ticket numbers  are unique  as there are  929 unique items and also it is an Object datatype so we can remove ticket feature
T_Concat2.Ticket.nunique()

In [ ]:
T_Concat2=T_Concat2.drop('Ticket',axis=1)
T_Concat2.info()

In [ ]:
#Now  we can split name string using name to  know  about the titles  of the travellers name in the ship.
##With the help of  titles , we can check if any priority perople  were present in ship.
###The priority members to be rescued first like  Children, females, doctors, lecturers  etc
#from nameparser import HumanName
#from nameparser.config import CONSTANTS
#CONSTANTS.string_format = "{last} {suffix} {title} {suffix} {first} {middle} ({nickname})"
#name=numpy.empty(1309,dtype='object')
#Title=numpy.empty(1309,dtype='object')
#for i in range(1,1310):
#    name[i-1]=HumanName(T_Concat2.loc[i,'Name'])
#    Title[i-1]=name[i-1].title
#Title.shape
#Title_df=pandas.DataFrame(Title, columns=['Title'])
#Title_df=Title_df.set_index(T_Concat2.index)
#Title_df.info()
#Title_df.value_counts()
#Title_df.isnull().sum()


In [ ]:
#Nameparser  is unable to identify  complete list of Titles, so we may use split function
name=numpy.empty(1309,dtype='object')
Title=numpy.empty(1309,dtype='object')
for i in range(1,1310):
    name[i-1]=T_Concat2.loc[i,'Name'].split(',')[1].split('.')[0].strip()
    Title[i-1]=name[i-1]
Title
Title_df=pandas.DataFrame(Title, columns=['Title'])
Title_df=Title_df.set_index(T_Concat2.index)
Title_df.info()
Title_df.value_counts()

In [ ]:
#Now  we need to classify titles to segragate  priority & priviledged titles Like Doctors, Children, Hon'ble People , Ex Defence Personnel etc in the observations
###Mr , Mrs, Dona,  to be classified as  General Title
####Miss , Mlle, Mme, Ms, Master to be classified as Single Title (Unmarried or Separated)
#####Lady, Sir,Jonkheer, Rev , Don , the Countess to be classified as Honble People
######Col, Major, Capt, to be classified as Defence People
#######Dr to be classified as Doctors
########Now  we will be classifying the Titles as per above classification
Title_df.Title = Title_df.Title.replace(['Sir','Lady','Don','Jonkheer','Rev','the Countess'], 'Honble People')
Title_df.Title = Title_df.Title.replace(['Mr','Mrs','Dona'], 'General People')
Title_df.Title = Title_df.Title.replace(['Miss','Mlle','Mme','Ms','Master'], 'Single People')
Title_df.Title = Title_df.Title.replace(['Col','Major','Capt'], 'Defense People')
Title_df.Title = Title_df.Title.replace(['Dr'], 'Doctor')
Title_df.value_counts()

In [ ]:
#Now  we wil apply one hot encoding to convert the categories to Numerical values
Title_En=encoder.fit_transform(Title_df.values)
Title_En
Title_En_Df=pandas.DataFrame(Title_En, columns=['Defense People','Doctor','General People','Honble People','Single/Children'])
Title_En_Df.head()

In [ ]:
Title_En_Df=Title_En_Df.set_index(T_Concat1.index)
T_Concat3=pandas.concat([T_Concat2,Title_En_Df],axis=1,sort=False)
T_Concat3.info()
T_Concat3.head()

In [ ]:
#Now we ca remove  Name variable  from the dataset and it will be our final concatenated dataset used for modelling
T_final=T_Concat3.drop('Name',axis=1)
T_final.info()

In [ ]:
#Now we would start modelling of the dataset by spot checking various algorithms for best accuracy
#First we would scale the Fare and Age variable with normalization as  these are continous variables in the dataset
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
T_final[['Age','Fare']] = scaler.fit_transform(T_final[['Age','Fare']].values)
T_final.head()

In [ ]:
#Now can scale PClass as well with normalization 
T_final[['Pclass']] = scaler.fit_transform(T_final[['Pclass']].values)
T_final.head()

In [ ]:
#Now we  need to separate  concatenated dataset  to original test and training dataset
Test_Final=T_final.loc[892:]
Test_Final.head()

In [ ]:
Train_Final=T_final.loc[:891]
Train_Final.tail()

In [ ]:
#Checking whether the dataset is imbalanced. It seems data set  is quite  balanced
Train_Final['Survived'].value_counts()

In [ ]:
#As the dataset is slight imbalanced, so we would spot check various algorithms to check accuracy on training dataset
array=Train_Final.values
Train_X = array[:,1:16]
Train_Y = array[:,0]
models = []
models.append(('LR', LogisticRegression(solver='liblinear')))
models.append(('LDA', LinearDiscriminantAnalysis()))
models.append(('NB', GaussianNB()))
models.append(('RC', RidgeClassifier()))
models.append(('SVC', SVC()))
#Below algorithms are bagging and boosting ensemble  methods to check if the performance of model is increased
models.append(('Random Forest',RandomForestClassifier(n_estimators=100, max_features=9)))
models.append(('Extra Trees',ExtraTreesClassifier(n_estimators=100, max_features=9)))
models.append(('Gradient Boosting',GradientBoostingClassifier(n_estimators=100,learning_rate=0.5,max_features=9)))
# evaluate each model in turn
results = []
names = []

scoring = 'accuracy'
for name, model in models:
    kfold = KFold(n_splits=10, random_state=7,shuffle=True)
    cv_results = cross_val_score(model, Train_X, Train_Y, cv=kfold, scoring=scoring)
    results.append(cv_results.mean())
    names.append(name)
    print('%s: %f (%f)' % (name, cv_results.mean(), cv_results.std()))
#Now  we can plot  accuracy metric  against methods  used in order to analze  which model is better
from matplotlib import pyplot
pyplot.figure(figsize=(10,10))
seaborn.boxplot(x=names, y=results,palette='flare')

In [ ]:
#Accuracy  of SVC, Randon Forest and Gradient Boost  are  good. 
###Now  we would check other  metrices  like precision, recall and f1-score
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import roc_curve
from sklearn.metrics import RocCurveDisplay
from sklearn.metrics import classification_report
models = []
models.append(('LR', LogisticRegression(solver='liblinear')))
models.append(('LDA', LinearDiscriminantAnalysis()))
models.append(('NB', GaussianNB()))
models.append(('RC', RidgeClassifier()))
models.append(('SVC', SVC()))
#Below algorithms are bagging and boosting ensemble  methods to check if the performance of model is increased
models.append(('Random Forest',RandomForestClassifier(n_estimators=100, max_features=9)))
models.append(('Extra Trees',ExtraTreesClassifier(n_estimators=100, max_features=9)))
models.append(('Gradient Boosting',GradientBoostingClassifier(n_estimators=100,learning_rate=0.5,max_features=9)))
# evaluate each model in turn
predict_list = []
names = []
n=0
for name, model in models:
    kfold = KFold(n_splits=10, random_state=7,shuffle=True)
    y_pred = cross_val_predict(model, Train_X, Train_Y, cv=3)
    predict_list.append(y_pred)
    names.append(name)
    cm=confusion_matrix(Train_Y, predict_list[n],labels=[0,1])
    print('Classification report  of Model',name)
    print(classification_report(Train_Y, predict_list[n], target_names=['class 0','class 1']))
    n=n+1
  

In [ ]:
#Model Selection - As  we can see the  other matrices likepredict , recall and f1 score  are good for Algorthm SVC
##Let us select SVC and try to improve results
###We can now fine tune some of the parameters like max features, n_estimators and learning rate
from sklearn.model_selection import GridSearchCV
param_grid = dict(gamma=["auto","scale"],kernel = ["linear", "rbf", "poly"])
grid = GridSearchCV(estimator=SVC(), param_grid=param_grid,cv=10)
grid.fit(Train_X, Train_Y)
print('LR Accuracy using Grid Search ', grid.best_score_)
print('LR Best Parameters using Grid Search', grid.best_params_)

In [ ]:
#Now we would check different values of degree when kernel is set to 'Poly' as Kernel as 'Poly' is best  kernel parameter
degree = [0, 1, 2, 3, 4, 5, 6]
param_grid = dict(degree = [0, 1, 2, 3, 4, 5, 6])
grid = GridSearchCV(estimator=SVC(gamma='scale',kernel='poly'), param_grid=param_grid,cv=10)
grid.fit(Train_X, Train_Y)
print('LR Accuracy using Grid Search ', grid.best_score_)
print('LR Best Parameters using Grid Search', grid.best_params_)

In [ ]:
#Seems no change in accuracy level, so now  we would  SVC as algorithm for  final model with gamma': 'scale', 'kernel': 'poly'
##Now  we would save the model and apply  the model on test data  to predict the outcome
Final_model_Titanic=SVC(gamma='scale',kernel='poly',degree=3)
Final_model_Titanic.fit(Train_X, Train_Y)
filename = 'Titanic_Survivor_Predict.sav'
pickle.dump(Final_model_Titanic, open(filename, 'wb'))

In [ ]:
#Now we would  calcuate predicted values on test data
predict_final=pandas.DataFrame(data = Final_model_Titanic.predict(Test_Final.values[:,1:16]),index=Test_Final.index)

In [ ]:
#We will format the predictions as requested in the problem
predict_final=predict_final.astype('int64')
predict_final.rename(columns={0:"Survived"},inplace=True)
predict_final.reset_index(inplace=True)
predict_final.head()

In [ ]:
#Converting the final result / predictions to csv file
predict_final.to_csv("Titanic_predicted_values.csv",index=False)